In [ ]:
# ── 파일 구조 미리보기 (컬럼 확인용) ─────────────────────────────────────────
import pandas as pd
from pathlib import Path

# ▼ 확인할 파일 지정
target_file = r"C:\Users\Perovskite\Documents\afm-analysis\output\NG\NG2_p2_pos1__220.txt"
skip_rows   = 3   # 데이터 앞 건너뛸 행 수 (헤더 포함 이전 줄 수)

# 원시 상위 10행 출력
print("=== 파일 원시 내용 (상위 10줄) ===")
with open(target_file, encoding='utf-8', errors='replace') as f:
    for i, line in enumerate(f):
        if i >= 10: break
        print(f"  [{i}] {line}", end='')

print("\n\n=== pandas 로드 결과 ===")
df = pd.read_csv(target_file, sep='\t', skiprows=skip_rows, header=0,
                 encoding='utf-8', errors='replace')
print(f"컬럼 수: {len(df.columns)}")
print(f"컬럼명: {list(df.columns)}")
print(df.head(3).to_string())


In [ ]:
# ── X/Y 컬럼 선택 → 2-컬럼 txt 저장 (단일 파일) ──────────────────────────────
# 위 미리보기 셀로 컬럼 구조 확인 후 아래 값을 설정

folder    = r"C:\Users\Perovskite\Documents\afm-analysis\output\NG\two_col_dat"
target_file = str(Path(folder) / (f_stems[0] + ".txt"))  # f_stems 는 위 셀에서 정의

# two_col_dat 파일 구조:
#   Row 0: "point XX"       ← skip
#   Row 1: Index  PiFM Amplitude  PiFM Phase  PiFM Wavenumber ...  ← 헤더
#   Row 2: (단위행)          ← skip
#   Row 3+: 데이터
skip_rows = [0, 2]   # 0번(제목), 2번(단위행) 건너뜀 → Row 1이 헤더

x_col       = 3    # PiFM Wavenumber (0=Index, 1=Amplitude, 2=Phase, 3=Wavenumber, ...)
y_col       = 1    # PiFM Amplitude

x_col_name  = "Wavenumber"
y_col_name  = "PiFM Amplitude"
x_unit      = "cm-1"
y_unit      = "uV"

out_file    = None   # None 이면 원본 파일명_2col.txt

# ─────────────────────────────────────────────────────────────────────────────
df = pd.read_csv(target_file, sep='\t', skiprows=skip_rows, header=0,
                 encoding='utf-8', errors='replace')

print("컬럼명:", list(df.columns))
print(df.head(3).to_string())

x_data = df.iloc[:, x_col] if isinstance(x_col, int) else df[x_col]
y_data = df.iloc[:, y_col] if isinstance(y_col, int) else df[y_col]

x_data = pd.to_numeric(x_data, errors='coerce')
y_data = pd.to_numeric(y_data, errors='coerce')
mask   = x_data.notna() & y_data.notna()
x_data, y_data = x_data[mask].values, y_data[mask].values

out_path = out_file or str(Path(target_file).with_suffix('')) + "_2col.txt"
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(f"{x_col_name}\t{y_col_name}\n")
    f.write(f"{x_unit}\t{y_unit}\n")
    for xv, yv in zip(x_data, y_data):
        f.write(f"{xv}\t{yv}\n")

print(f"\n저장 완료: {Path(out_path).name}  ({len(x_data)} pts)")
print(f"X 범위: {x_data[0]:.4f} ~ {x_data[-1]:.4f}")
print(f"Y 범위: {y_data.min():.6g} ~ {y_data.max():.6g}")


In [ ]:
# ── 폴더 내 전체 일괄 변환 ───────────────────────────────────────────────────
folder     = r"C:\Users\Perovskite\Documents\afm-analysis\output\NG\two_col_dat"
pattern    = "*.txt"
skip_rows  = [0, 2]   # 제목행(0), 단위행(2) 건너뜀
x_col      = 3        # PiFM Wavenumber
y_col      = 1        # PiFM Amplitude
x_col_name = "Wavenumber"
y_col_name = "PiFM Amplitude"
x_unit     = "cm-1"
y_unit     = "uV"
out_suffix = "_2col"

# ─────────────────────────────────────────────────────────────────────────────
txt_files = sorted(Path(folder).glob(pattern))
print(f"대상: {len(txt_files)}개")

ok, fail = 0, 0
for f in txt_files:
    if f.stem.endswith(out_suffix):   # 이미 변환된 파일 건너뜀
        continue
    try:
        df     = pd.read_csv(f, sep='\t', skiprows=skip_rows, header=0,
                              encoding='utf-8', errors='replace')
        x_data = pd.to_numeric(df.iloc[:, x_col] if isinstance(x_col, int) else df[x_col], errors='coerce')
        y_data = pd.to_numeric(df.iloc[:, y_col] if isinstance(y_col, int) else df[y_col], errors='coerce')
        mask   = x_data.notna() & y_data.notna()
        x_data, y_data = x_data[mask].values, y_data[mask].values

        out_path = str(f.with_suffix('')) + out_suffix + ".txt"
        with open(out_path, 'w', encoding='utf-8') as fo:
            fo.write(f"{x_col_name}\t{y_col_name}\n")
            fo.write(f"{x_unit}\t{y_unit}\n")
            for xv, yv in zip(x_data, y_data):
                fo.write(f"{xv}\t{yv}\n")

        print(f"  {f.name}  ->  {Path(out_path).name}  ({len(x_data)} pts)")
        ok += 1
    except Exception as e:
        print(f"  오류 [{f.name}]: {e}")
        fail += 1

print(f"\n완료: {ok}개 성공  /  {fail}개 실패")


---
## SpectroChemPy 활용 변환

SpectroChemPy()는 SPA **읽기**는 지원하나 **쓰기는 미지원**.
대신 (JCAMP-DX )로 OMNIC 호환 파일 생성 — 수동 구현보다 검증된 방법.

In [26]:
# ── SpectroChemPy 설치 확인 ─────────────────────────────────────────────
# 미설치 시: pip install spectrochempy
import spectrochempy as scp
import numpy as np
from pathlib import Path

print("spectrochempy version:", scp.__version__)

spectrochempy version: 0.10.1


In [34]:
# file folder
path_name = r"C:\Users\Perovskite\Documents\afm-analysis\output\NG"

In [47]:
# 폴더 내 모든 txt 파일
txt_files = sorted(Path(path_name).glob("*.txt"))
f_stems = [f.stem for f in txt_files]
f_stems

['NG2_p2_pos1__220',
 'NG2_p2_pos2_435',
 'NG2_p4_pos2_sb_l2.0_points_260601__280_02',
 'NG2_p4_pos2_sb_l2.0_points_260601__280_12',
 'NG2_p4_pos3_sb_l2.0_points_260601__383_01',
 'NG2_p4_pos3_sb_l2.0_points_260601__383_06',
 'NG2_pos1_sb_12.0_points_260601__152_01',
 'NG2_pos1_sb_12.0_points_260601__152_08',
 'NG2_pos2_sb_l2.0_points_260601__215_04',
 'result_430_amplitude_uV']

In [50]:
# ── txt → NDDataset → JCAMP-DX 변환 (단일 파일) ──────────────────────────
#in_file  = r"../output/spc_convert_test/result_261_avg_amplitude_uV.txt"
in_file = r"../output/NG/" + f_stems[0] + ".txt"
out_base = r"../output/NG/" + f_stems[0]   # .jdx 자동 추가

x, ys, names, units_list = read_txt_o(in_file)

for i, y in enumerate(ys):
    y_name = names[i+1] if i+1 < len(names) else f"Col{i+1}"
    y_unit = units_list[i+1] if i+1 < len(units_list) else ""

    # write_jcamp은 2D 데이터셋 (1, n) + y/x 두 좌표 필요
    wn_coord  = scp.Coord(x.astype(float), units="cm^-1", title="wavenumber")
    row_coord = scp.Coord([0], title="spectra")
    ds = scp.NDDataset(
        y.astype(float).reshape(1, -1),
        coordset=scp.CoordSet(y=row_coord, x=wn_coord),
        title=y_name,
    )

    suffix = f"_{i+1:02d}" if len(ys) > 1 else ""
    out_path = out_base + suffix + ".jdx"

    ds.write_jcamp(out_path, overwrite=True)
    print(f"{y_name}  ->  {Path(out_path).name}")


Kilo  ->  NG2_p2_pos1__220.jdx


In [44]:
# ── 다중 컬럼 txt → 개별 .jdx 일괄 변환 ────────────────────────────────────
batch_dir = Path("../output/NG")
txt_files = sorted(batch_dir.glob("*.txt"))

ok, fail = 0, 0
for txt_file in txt_files:
    try:
        x, ys, names, units_list = read_txt_o(txt_file)
        for i, y in enumerate(ys):
            y_name    = names[i+1] if i+1 < len(names) else f"Col{i+1}"
            wn_coord  = scp.Coord(x.astype(float), units="cm^-1", title="wavenumber")
            row_coord = scp.Coord([0], title="spectra")
            ds = scp.NDDataset(
                y.astype(float).reshape(1, -1),
                coordset=scp.CoordSet(y=row_coord, x=wn_coord),
                title=y_name,
            )
            suffix   = f"_{i+1:02d}" if len(ys) > 1 else ""
            out_path = str(txt_file.with_suffix("")) + suffix + ".jdx"
            ds.write_jcamp(out_path, overwrite=True)
            ok += 1
    except Exception as e:
        print(f"오류 [{txt_file.name}]: {e}")
        fail += 1

print(f"\n완료: {ok}개 .jdx 생성  /  {fail}개 실패")



완료: 48개 .jdx 생성  /  0개 실패


In [ ]:
# ── 생성된 .jdx 파일을 scp로 다시 읽어 확인 ────────────────────────────────
# scp.read_jcamp 은 상대경로를 GitHub에서 찾으므로 절대경로 필요
import os
jdx_file = os.path.abspath(r"../output/085/result_085_avg.jdx")

ds_check = scp.read_jcamp(jdx_file)
print(ds_check)

import matplotlib.pyplot as plt
ds_check.plot()
plt.show()
